In [30]:
import sys
from opentrons import protocol_api, execute, simulate
from opentrons.types import Location, Point
import pandas as pd
import numpy as np
import urllib.request
import time
import ast  # safer than eval

## Run this in terminal on Shared Folder on Computer side! (NOT MARA)

```
cd: Mara_shared
python -m http.server 8000
```

This will open a sever to allow opentron to acess the file from: 

csv_path = "http://192.168.0.100:8000/conditions.csv"   if on Mara Laptop\
local_path = "http://127.0.0.1:8000/conditions.csv" if on local Laptop

In [32]:
path = "http://192.168.0.102:8000/conditions_save.csv" # mara Laptop
# path = "http://127.0.0.1:8000/conditions_save.csv" # for local testing
# path = 'http://192.168.0.110:8000/conditions_save.csv'
df = pd.read_csv(path)[-8:]
df


,id,well,volC,volM,volY,volK
0,1,A1,22.2,35.1,27.3,15.3
1,2,B1,18.6,22.9,54.3,4.3
2,3,C1,49.5,17.7,4.8,28.0
3,4,D1,30.6,28.0,5.8,35.6
4,5,E1,3.0,36.8,16.4,43.7
5,6,F1,17.0,31.3,19.7,31.9
6,7,G1,13.6,11.2,37.6,37.6
7,8,H1,41.2,7.4,39.5,11.9


In [ ]:
# metadata
metadata = {
    "protocolName": "IMOD Protocol",
    "author": "Clara Tamura, Austin Martin",
    "description": "Mara OT-2 Protocol for IMOD project",
    "apiLevel": "2.19",
}

class OpentronMara:
    def __init__(self, protocol):
        self.protocol = protocol
        self.protocol._commands.clear()

        # Load tipracks
        self.tiprack_300 = protocol.load_labware('opentrons_96_tiprack_300ul', '5')

        # Load Pipettes
        self.pip300 = protocol.load_instrument('p300_single_gen2', mount='right', tip_racks=[self.tiprack_300])

        # Track used tips manually
        self.tip_log = {
            self.pip300: {'tips': self.tiprack_300.wells(), 'next': 0},
        }

        # Set pipette buffers
        self.pip300_buffer = 20

        self.pip300.flow_rate.aspirate = 50
        self.pip300.flow_rate.dispense = 50
        self.pip300.flow_rate.blow_out = 100

        # Load plates
        self.plate_96 = protocol.load_labware('opentrons_96_wellplate_200ul_pcr_full_skirt', '1')
        self.plate_96.set_offset(x=0, y=0, z=120)  # Z lowered by 1 mm
    
        self.plate_6 = protocol.load_labware('corning_6_wellplate_16.8ml_flat','3')
        self.plate_6.set_offset(x=0, y=0, z=5)
        self.source_wells = ['A1', 'A2', 'B1', 'B2']

        # Home the pipettes
        self.pip300.home()

        print("Initialization complete.")

    def pick_up_tip(self, pipette):
        """Pick up the next available tip for the given pipette, with rack limit check."""
        next_tip_index = self.tip_log[pipette]['next']
        tips = self.tip_log[pipette]['tips']
        
        if next_tip_index >= len(tips):
            raise RuntimeError(f"❌ No more tips available for {pipette.name}. Tip rack is empty!")

        tip = tips[next_tip_index]
        pipette.pick_up_tip(tip)
        self.tip_log[pipette]['next'] += 1

    def reset_tips(self, pipette=None):
        if pipette:
            self.tip_log[pipette]['next'] = 0
        else:
            for pip in self.tip_log:
                self.tip_log[pip]['next'] = 0

    def mixing(self, df: pd.DataFrame, columns):
        '''
        stock: array of stock locations 
        well: array of well locations
        volume: array of volumes to aspirate and dispense
        '''
        # Get the data from the CSV file
        self.columns = columns
        volumes_df = df[self.columns].round(1)
        num_samples, num_solutions = volumes_df.shape
        
        well_names = df['well']
        for i in range(num_solutions):
            max_volume = 200  # pipette capacity in µL
            self.pip300.pick_up_tip()
            for vol, dest_well in zip(volumes_df[self.columns[i]].to_list(), well_names):
                source_well = self.plate_6.wells(self.source_wells[i])[0]
                destination = self.plate_96.well(dest_well)

                while vol > 0:
                    aspirate_vol = min(vol, max_volume)
                    
                    self.pip300.aspirate(aspirate_vol, source_well.bottom(15))

                    # Wait at the top of the well for 3 seconds to let the drops fall
                    self.pip300.move_to(source_well.top(5))
                    self.protocol.delay(seconds=3)

                    self.pip300.move_to(destination.top(-1))
                    self.pip300.dispense(aspirate_vol,destination.top(-1))
                    
                    self.pip300.blow_out(destination.top(-1))
                    #self.pip300.touch_tip(destination)

                    vol -= aspirate_vol  # reduce the remaining volume to transfer

            self.pip300.return_tip()
        
        #self.print_cmd()
        
    def distribute_colors(self, df: pd.DataFrame, columns):
        self.columns = columns
        volumes_df = df[self.columns].round(1)
        num_samples, num_solutions = volumes_df.shape
        well_names = df['well']
        destination = [self.plate_96.wells(well) for well in well_names]
        for i in range(num_solutions):
            self.pip300.distribute(
                volumes_df[self.columns[i]].to_list(),
                self.plate_6.wells(self.source_wells[i])[0],
                destination,
                disposal_vol=0,
                blow_out=True,
                blowout_location='source well',
                new_tip='once',
                trash=False
            )
        self.reset_tips(self.pip300) # Reset the tip log for the pipette after use
        #self.print_cmd()
        
        
    def print_cmd(self):
        print("Printing Opentron Commands:")
        for cmd in self.protocol.commands():
            print(cmd)
    
    def clear_protocol(self):
        self.protocol._commands.clear()




In [50]:
protocol = execute.get_protocol_api('2.19') 
#protocol = simulate.get_protocol_api('2.19')  # Use simulate for testing
robot = OpentronMara(protocol)

Initialization complete.


In [33]:
# robot.pip300.drop_tip()  # Ensure the pipette is not holding a tip before starting

In [48]:
robot.pip300.return_tip()

<InstrumentContext: p300_single_v2.1 in RIGHT>

In [96]:
robot.protocol._commands.clear()
robot.tiprack_300.reset()  # Reset the tip racks to ensure a clean state

In [97]:
df = pd.read_csv(path)[-8:]
df

,id,well,volC,volM,volY,volK
48,49,A7,50.0,50.0,0.0,0.0
49,50,B7,33.3,33.3,0.0,33.3
50,51,C7,0.0,50.0,50.0,0.0
51,52,D7,42.1,15.8,42.1,0.0
52,53,E7,0.0,0.0,100.0,0.0
53,54,F7,0.0,33.3,33.3,33.3
54,55,G7,33.3,0.0,33.3,33.3
55,56,H7,0.0,0.0,0.0,100.0


In [ ]:
robot.distribute_colors(df, ['volC', 'volM', 'volY', 'volK'])

In [ ]:
# Mix the LHS sampled data --> Only do this once
# robot.mixing(df, ['volC', 'volM', 'volY', 'volK'])

KeyboardInterrupt: 

In [ ]:
processed_ids = set()  # This should persist during a single script run

while True:
    try:
        robot.protocol._commands.clear()
        robot.clear_protocol()
        df = pd.read_csv(path)[-8:]  # Read the last row of the CSV file
        print(df)
        # Extract volumes into a list of tuples
        # print(volumes[0][0],volumes[0][1],volumes[0][2])

        if df['id'].values[0] > last_id:
            robot.mixing(df, ['volC', 'volM', 'volY', 'volK'])
            robot.protocol._commands.clear()
            last_id = df['id'][-1:].values[0]
            # if "exit" in df['id'].values:
            #     print("exit success")
            #     break

        time.sleep(20)

    except Exception as e:
        print("Error reading file", e)
        break


   id well  volC  volM  volY  volK
4   4   A5  77.1  95.2  59.9  17.7
   id well  volC  volM  volY  volK
4   4   A5  77.1  95.2  59.9  17.7
   id well  volC  volM  volY  volK
5   5   A6  10.0  20.0  30.0  40.0
Printing Opentron Commands:
Picking up tip from E1 of Opentrons OT-2 96 Tip Rack 300 µL on slot 7
Aspirating 10.0 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 3 at 50.0 uL/sec
Moving to A1 of Corning 6 Well Plate 16.8 mL Flat on slot 3
Delaying for 0 minutes and 3.0 seconds
Moving to A6 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1
Dispensing 10.0 uL into A6 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1 at 50.0 uL/sec
Blowing out at A6 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1
Touching tip
Returning tip
Dropping tip into E1 of Opentrons OT-2 96 Tip Rack 300 µL on slot 7
Picking up tip from F1 of Opentrons OT-2 96 Tip Rack 300 µL on slot 7
Aspirating 20.0 uL from A2 of Corning 6 Well Plate 16.8 mL Flat on slot